# 02 · Genie Agent + Genie One

Build a trusted Genie Agent on the workshop tables, then schedule the Monday question in Genie One.

Everything you need to paste into the Genie UI is **in this notebook** — tables, joins, instructions, and example SQL with your real catalog/schema.

**Ends with:** a Genie One scheduled report (Lab B).


In [ ]:
%run ./00_setup


## Known-good queries

Duplicate = later claim for the same `pet_id + clinic_id + visit_date`. Invoice number is **not** the key.
Sibling pets on the same day are **not** duplicates (trap: household **PH0013**).


In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW v_suspected_duplicate_claims AS
WITH ranked AS (
  SELECT c.*,
    ROW_NUMBER() OVER (PARTITION BY pet_id, clinic_id, visit_date ORDER BY submitted_date, claim_id) AS visit_rank,
    FIRST_VALUE(claim_id) OVER (PARTITION BY pet_id, clinic_id, visit_date ORDER BY submitted_date, claim_id) AS original_claim_id,
    FIRST_VALUE(claim_status) OVER (PARTITION BY pet_id, clinic_id, visit_date ORDER BY submitted_date, claim_id) AS original_status
  FROM claims c
)
SELECT * FROM ranked WHERE visit_rank > 1
""")
print("view ready")


In [ ]:
display(spark.sql("""
SELECT
  COUNT(*) AS suspected_duplicates,
  COUNT_IF(d.claim_status = 'Paid') AS duplicates_paid,
  COUNT_IF(d.claim_status = 'Denied' AND d.denial_reason = 'Duplicate claim') AS duplicates_caught,
  ROUND(SUM(p.payment_amount), 2) AS leakage_usd
FROM v_suspected_duplicate_claims d
LEFT JOIN claim_payments p USING (claim_id)
"""))


In [ ]:
display(spark.sql("""
SELECT v.clinic_name, COUNT(*) AS suspected_duplicates
FROM v_suspected_duplicate_claims d
JOIN vet_clinics v USING (clinic_id)
GROUP BY v.clinic_name
ORDER BY suspected_duplicates DESC
LIMIT 10
"""))


In [ ]:
# Trap: PH0013 — sibling pets same clinic/day. Correct Genie answer: no duplicates.
display(spark.sql("""
SELECT c.claim_id, c.pet_id, p.policyholder_id, c.clinic_id, c.visit_date, c.claim_status
FROM claims c
JOIN pets p USING (pet_id)
WHERE p.policyholder_id = 'PH0013'
ORDER BY c.visit_date, c.pet_id
"""))


## Metric view


In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW claims_metrics
WITH METRICS LANGUAGE YAML AS $$
version: 1.1
source: claims
joins:
  - name: policies
    source: policies
    on: source.policy_id = policies.policy_id
dimensions:
  - name: submitted_month
    expr: DATE_TRUNC('MONTH', submitted_date)
  - expr: diagnosis_category
  - expr: claim_status
  - expr: submission_channel
  - name: plan_tier
    expr: policies.plan_tier
measures:
  - name: claim_count
    expr: COUNT(DISTINCT claim_id)
  - name: approval_rate
    expr: COUNT_IF(claim_status IN ('Approved','Paid')) / NULLIF(COUNT_IF(claim_status IN ('Approved','Paid','Denied')), 0)
    format:
      type: percentage
  - name: denial_rate
    expr: COUNT_IF(claim_status = 'Denied') / NULLIF(COUNT_IF(claim_status IN ('Approved','Paid','Denied')), 0)
    format:
      type: percentage
  - name: avg_days_to_decision
    expr: AVG(DATEDIFF(decision_date, submitted_date))
    synonyms: [processing time, time to settle]
  - name: open_claims
    expr: COUNT_IF(claim_status IN ('Submitted','In Review'))
  - name: total_claimed
    expr: SUM(claimed_amount)
    format:
      type: currency
      currency_code: USD
$$
""")
display(spark.sql("""
SELECT diagnosis_category, MEASURE(approval_rate), MEASURE(avg_days_to_decision)
FROM claims_metrics
GROUP BY ALL
ORDER BY 1
"""))


## Build the Genie Agent (UI)

1. **New → Genie Agent**
2. Add tables (run the next cell for the exact names):
   - `claims`, `claim_payments`, `policies`, `pets`, `policyholders`, `vet_clinics`
   - optional: `claims_metrics`, `earned_premium_monthly`
3. **Joins** — if PK/FKs from notebook 01 did not appear automatically:
   **Configure → Examples → Add → Joins**, then add:
   - claims → policies on `policy_id`
   - claims → pets on `pet_id`
   - claims → vet_clinics on `clinic_id`
   - claim_payments → claims on `claim_id`
   - pets → policyholders on `policyholder_id`
4. Paste **instructions** and **example SQL** from the cells below (self-contained CTEs — Genie cannot see `v_suspected_duplicate_claims` unless you add that view).


In [ ]:
print("=== Tables to add ===")
for t in [
    "claims", "claim_payments", "policies", "pets", "policyholders", "vet_clinics",
    "claims_metrics", "earned_premium_monthly",
]:
    print(f"  {CATALOG}.{SCHEMA}.{t}")

instructions = f"""This agent analyzes synthetic pet insurance claim data. All amounts are USD.

Duplicate (double) claim = a second or later claim for the same vet visit: same pet_id, clinic_id, visit_date.
The earliest submitted claim is the original. Do NOT rely on invoice_number or claimed_amount;
resubmissions reuse, suffix, or change them. Two different pets from one household on the same day are NOT duplicates.

Leakage = suspected duplicate with status Paid. Caught = denied with reason 'Duplicate claim'.
Open = Submitted or In Review. Decided = Approved, Paid, or Denied.
Approval and denial rates use decided claims only; show as percentages.
Processing time = average days from submitted_date to decision_date.
Reimbursement rate is a decimal; show it as a percentage.
"""
print("\n=== Instructions (paste) ===\n")
print(instructions)

fq = f"{CATALOG}.{SCHEMA}"
q1 = f"""WITH ranked AS (
  SELECT c.*,
    ROW_NUMBER() OVER (PARTITION BY pet_id, clinic_id, visit_date ORDER BY submitted_date, claim_id) AS visit_rank
  FROM {fq}.claims c
)
SELECT COUNT(*) AS suspected_duplicates,
  COUNT_IF(d.claim_status = 'Paid') AS duplicates_paid,
  ROUND(SUM(p.payment_amount), 2) AS leakage_usd
FROM ranked d
LEFT JOIN {fq}.claim_payments p USING (claim_id)
WHERE visit_rank > 1
"""
q2 = f"""WITH ranked AS (
  SELECT c.*,
    ROW_NUMBER() OVER (PARTITION BY pet_id, clinic_id, visit_date ORDER BY submitted_date, claim_id) AS visit_rank
  FROM {fq}.claims c
)
SELECT v.clinic_name, COUNT(*) AS suspected_duplicates
FROM ranked d
JOIN {fq}.vet_clinics v USING (clinic_id)
WHERE visit_rank > 1
GROUP BY v.clinic_name
ORDER BY suspected_duplicates DESC
LIMIT 10
"""
q3 = f"""WITH ranked AS (
  SELECT c.*, p.policyholder_id,
    ROW_NUMBER() OVER (PARTITION BY c.pet_id, c.clinic_id, c.visit_date ORDER BY c.submitted_date, c.claim_id) AS visit_rank
  FROM {fq}.claims c
  JOIN {fq}.pets p USING (pet_id)
  WHERE p.policyholder_id = 'PH0013'
)
SELECT claim_id, pet_id, clinic_id, visit_date, claim_status
FROM ranked
WHERE visit_rank > 1
"""
print("\n=== Example SQL 1 — leakage ===\n")
print(q1)
print("\n=== Example SQL 2 — clinics ===\n")
print(q2)
print("\n=== Example SQL 3 — trap PH0013 (expect 0 rows) ===\n")
print(q3)


## Benchmark questions

1. How many suspected duplicate claims are there, and how much did we pay on them?
2. Which clinics have the most double submissions?
3. What is the approval rate by diagnosis category?
4. What is the average processing time by submission channel?
5. What is the loss ratio by plan tier for 2025?
6. **Trap:** Show duplicates for household **PH0013**. Expected: **none** (sibling pets, not duplicates).


## Your turn — Lab A · Genie Agent (15 min)

Add your own instruction that fixes one wrong answer. Rerun a benchmark question and confirm the answer changed.

Suggested fail: ask *Show duplicates for household PH0013* on an agent **without** the sibling-pet instruction, then add it.


## Your turn — Lab B · Genie One (10 min) — leave this running

In Genie One, ask:

> Every Monday at 8am, send me last week's suspected duplicate claims, how many were paid, total leakage, and approval rate by diagnosis.

Then **schedule** it. This is the automated outcome for Module 1.
